# ASAP8 Detection of Change analysis

Compact DoC notebook for optical-spike image variability and longitudinal image-response timing. Standalone anatomical depth analyses live in the separate depth notebook.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import ndimage
from matplotlib.lines import Line2D
from IPython.display import display, HTML

from vip_slap2_analysis.utils.utils import save_figure
from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.voltage.dataset import DEPTH_GROUP_ORDER, build_voltage_session_table, build_voltage_roi_table
from vip_slap2_analysis.voltage.spikes import DETECTOR_VERSION, build_spike_table
from vip_slap2_analysis.behavior.change_detection import build_change_detection_events
from vip_slap2_analysis.voltage.responses import build_single_trial_index, load_response_package, get_mean_response, get_sequence_response, load_single_trial_traces
from vip_slap2_analysis.behavior.encoder import compute_encoder_velocity

assert DETECTOR_VERSION == "template_v1"
sns.set_style("white")
plt.rcParams.update({"legend.fontsize":"x-large","axes.labelsize":"xx-large","axes.titlesize":"xx-large","xtick.labelsize":"xx-large","ytick.labelsize":"xx-large"})
display(HTML("<style>.container { width:100% !important; }</style>"))

# 1. Setup
## Configuration

In [ ]:
BASE_PATH = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")
SAVE_PATH = Path(r"C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Lab_Meetings\2026-07-28_OPhys_LabMeetingV\figures\voltage_plots")
TARGET_MICE = [852835, 863774]
TARGET_SESSION_LABELS = None
TARGET_SESSION_IDS = None
PARADIGMS = ["change_detection_passive"]
EXCLUDE_SESSION_TYPES = ["expression_check", "volume_imaging"]
TRACE_VARIANT = "dff_robust_f0_trial"
REGISTRATION_FILENAME = "roi_identity_registration.csv"
EXCLUDE_INVALID_ROIS = True
EXPECTED_F0_SMOOTH_SEC = 60.0
FORCE_SPIKE_RECOMPUTE = False
SPIKE_KWARGS = dict(height_sigma=3.0, template_sigma=3.5, prominence_sigma=0.5)
RUNNING_KWARGS = dict(wheel_radius_cm=4.69, encoder_units="ticks", ticks_per_revolution=8192, absolute_velocity=True)

SESSION_ORDER = ["A0","A1","A2","B0","B1","B2"]
IMAGE_WINDOW_S = (0.0, 0.25)
MIN_TRIALS_PER_IMAGE = 5
PEAK_WINDOW_S = (-0.25, 0.50)
PEAK_SMOOTH_MS = 10.0
DEPTH_COLORS = {"<100 µm":"#EBA287","100–150 µm":"#d1e2b0",">150 µm":"#7bbcd5"}
def depth_group_from_um(depth):
    depth=float(depth)
    return "<100 µm" if depth < 100 else ("100–150 µm" if depth <= 150 else ">150 µm")

## Canonical DoC tables

In [ ]:
registry = VIPSessionRegistry.from_basepath(BASE_PATH)
sessions = build_voltage_session_table(registry, subject_ids=TARGET_MICE, paradigms=PARADIGMS, exclude_session_types=EXCLUDE_SESSION_TYPES, session_labels=TARGET_SESSION_LABELS, session_ids=TARGET_SESSION_IDS, trace_variant=TRACE_VARIANT, expected_f0_smooth_sec=EXPECTED_F0_SMOOTH_SEC)
rois = build_voltage_roi_table(sessions, registration_filename=REGISTRATION_FILENAME, exclude_invalid_rois=EXCLUDE_INVALID_ROIS)
rois["depth_um"] = pd.to_numeric(rois["depth_um"], errors="coerce")
rois["depth_group"] = rois["depth_um"].map(depth_group_from_um)
spikes = build_spike_table(sessions, rois, force=FORCE_SPIKE_RECOMPUTE, detection_kwargs=SPIKE_KWARGS)
events = build_change_detection_events(sessions)
trial_index = build_single_trial_index(sessions, events)

print(f"{len(sessions)} sessions · {rois['included'].sum()} included ROI observations · {len(spikes):,} spikes · {len(events):,} image cycles")
display(sessions[["subject_id","session_id","session_label","session_order","dmd1_depth_um","dmd2_depth_um"]])
display(rois.loc[rois["included"],["subject_id","session_label","dmd","roi","depth_um","depth_group"]].drop_duplicates().sort_values(["depth_um","subject_id","session_label"]))

# 2. Image responses
## Image-response peak latency

In [ ]:
def smooth_finite(y, sigma):
    y = np.asarray(y,float).reshape(-1); good = np.isfinite(y)
    if good.sum() < 3: return np.full_like(y,np.nan)
    y = np.interp(np.arange(y.size),np.flatnonzero(good),y[good])
    return ndimage.gaussian_filter1d(y,sigma,mode="nearest")

selected = sessions[sessions["session_label"].astype(str).isin(SESSION_ORDER)]
available_labels = [s for s in SESSION_ORDER if s in set(selected["session_label"].astype(str))]
rows = []
for session in selected.itertuples(index=False):
    sid = str(session.session_id); pkg = load_response_package(session.mean_npz)
    for r in rois[(rois["session_id"].astype(str)==sid) & rois["included"].astype(bool)].itertuples(index=False):
        dmd, roi, key = int(r.dmd), int(r.roi), f"DMD{int(r.dmd)}"
        if key not in pkg: continue
        for image in pkg[key].get("image_identity",{}):
            try: t,y = get_mean_response(pkg,dmd=dmd,source_roi=roi,event_type="image",image_name=image)
            except (KeyError,IndexError,ValueError): continue
            t,y = np.asarray(t,float).reshape(-1),np.asarray(y,float).squeeze()
            if y.ndim != 1 or len(y) != len(t) or len(y) < 3: continue
            dt = np.nanmedian(np.diff(t))
            if not np.isfinite(dt) or dt <= 0: continue
            ys = smooth_finite(y,(PEAK_SMOOTH_MS/1000)/dt)
            keep = (t>=PEAK_WINDOW_S[0]) & (t<=PEAK_WINDOW_S[1]) & np.isfinite(ys)
            if not keep.any(): continue
            idx = np.flatnonzero(keep); p = idx[np.nanargmax(ys[keep])]
            rows.append(dict(subject_id=str(r.subject_id),session_id=sid,session_label=str(r.session_label),cell_id=str(r.cell_id),global_cell_id=str(getattr(r,"global_cell_id","")),manually_registered=bool(getattr(r,"manually_registered",False)),depth_um=float(r.depth_um),depth_group=str(r.depth_group),image_name=str(image),peak_latency_s=float(t[p]),peak_dff=float(ys[p])))

image_peak_latency = pd.DataFrame(rows)
if image_peak_latency.empty: raise RuntimeError("No image-response peaks extracted.")
keys = ["subject_id","session_id","session_label","cell_id","global_cell_id","manually_registered"]
cell_session_peak = image_peak_latency.groupby(keys,observed=True,dropna=False).agg(peak_latency_s=("peak_latency_s","median"),peak_latency_q25_s=("peak_latency_s",lambda x:np.nanquantile(x,.25)),peak_latency_q75_s=("peak_latency_s",lambda x:np.nanquantile(x,.75)),n_images=("image_name","nunique"),depth_um=("depth_um","median")).reset_index()
cell_session_peak["depth_group"] = np.select([cell_session_peak["depth_um"]<100,cell_session_peak["depth_um"]>150],["<100 µm",">150 µm"],default="100–150 µm")
tracked_peak = cell_session_peak[cell_session_peak["manually_registered"]].groupby(["subject_id","global_cell_id"],group_keys=False,observed=True).filter(lambda x:x["session_id"].nunique()>=2)

In [ ]:
def finish_axis(ax):
    sns.despine(ax=ax); ax.tick_params(axis="both",labelsize=11)
    for spine in ax.spines.values(): spine.set_linewidth(2)

xmap = {s:i for i,s in enumerate(available_labels)}; x = np.arange(len(available_labels))
fig,ax = plt.subplots(figsize=(5.2,3.6))
for (_,cell),g in tracked_peak.groupby(["subject_id","global_cell_id"],observed=True):
    g = g.assign(x=g["session_label"].map(xmap)).dropna(subset=["x"]).sort_values("x")
    if len(g)>1: ax.plot(g["x"],1000*g["peak_latency_s"],color=DEPTH_COLORS[str(g["depth_group"].iloc[0])],lw=.8,alpha=.16,zorder=1)
for group,g in tracked_peak.groupby("depth_group",observed=True):
    q = g.groupby("session_label")["peak_latency_s"].agg(median="median",q25=lambda z:np.nanquantile(z,.25),q75=lambda z:np.nanquantile(z,.75)).reindex(available_labels)
    valid=q["median"].notna().to_numpy(); xx=x[valid]; c=DEPTH_COLORS[str(group)]
    ax.fill_between(xx,1000*q["q25"].to_numpy()[valid],1000*q["q75"].to_numpy()[valid],color=c,alpha=.4,lw=0)
    ax.plot(xx,1000*q["median"].to_numpy()[valid],"-o",color=c,lw=3,ms=7,mec="black",mew=.8,label=str(group),zorder=3)
ax.axhline(0,color=".5",lw=1,ls="--")
if "A2" in xmap and "B0" in xmap: ax.axvline((xmap["A2"]+xmap["B0"])/2,color=".65",lw=1,ls=":")
ax.set(xticks=x,xticklabels=available_labels,xlabel="Session",ylabel="Time of maximum mean dF/F\nrelative to image onset (ms)",
#        ylim=(1000*PEAK_WINDOW_S[0],1000*PEAK_WINDOW_S[1]),
       title="Image-response timing across sessions")
ax.axhline(250,color='k',lw=0.5,dashes=[6,3],zorder=0)
finish_axis(ax); ax.legend(title="Depth",frameon=False,fontsize=9); fig.tight_layout()
save_figure(fig,os.path.join(SAVE_PATH,"latency_shift"),formats=[".pdf"],dpi=300); plt.show()

## Trial-wise image variability
Ordinary image presentations use 0–250 ms firing rate. `RMS²` is trial-count-weighted between-image firing-rate variance; `FVE = RMS² / total firing-rate variance`.

In [ ]:
ordinary_events = events[events["retained_image_window"].astype(bool) & ~events["is_change"].astype(bool) & ~events["is_omission"].astype(bool) & events["image_label"].notna()].copy()
spike_lookup = {(str(sid),int(dmd),int(roi)):np.sort(g["spike_time_sec"].to_numpy(float)) for (sid,dmd,roi),g in spikes.groupby(["session_id","dmd","roi"],observed=True)}
trial_rows=[]; start,stop=IMAGE_WINDOW_S; duration=stop-start
for session in sessions.itertuples(index=False):
    sid=str(session.session_id); ev=ordinary_events[ordinary_events["session_id"].astype(str)==sid][["event_id","onset_sec","image_label","time_block"]]
    onsets=ev["onset_sec"].to_numpy(float)
    for r in rois[(rois["session_id"].astype(str)==sid) & rois["included"].astype(bool)].itertuples(index=False):
        st=spike_lookup.get((sid,int(r.dmd),int(r.roi)),np.array([],float))
        rate=(np.searchsorted(st,onsets+stop,side="left")-np.searchsorted(st,onsets+start,side="left"))/duration
        t=ev.copy(); t["image_rate_hz"]=rate; t["subject_id"]=str(r.subject_id); t["session_id"]=sid; t["session_label"]=str(r.session_label); t["session_order"]=int(r.session_order); t["dmd"]=int(r.dmd); t["roi"]=int(r.roi); t["cell_id"]=str(r.cell_id); t["global_cell_id"]=str(getattr(r,"global_cell_id","")); t["manually_registered"]=bool(getattr(r,"manually_registered",False)); t["depth_um"]=float(r.depth_um); t["depth_group"]=str(r.depth_group)
        trial_rows.append(t)
image_trial_df=pd.concat(trial_rows,ignore_index=True)

metric_keys=["subject_id","session_id","session_label","session_order","dmd","roi","cell_id","global_cell_id","manually_registered","depth_um","depth_group"]
metric_rows=[]
for keys,t in image_trial_df.groupby(metric_keys,observed=True,dropna=False):
    counts=t["image_label"].value_counts(); keep=counts[counts>=MIN_TRIALS_PER_IMAGE].index; q=t[t["image_label"].isin(keep)]
    y=q["image_rate_hz"].to_numpy(float)
    if len(y)<2 or q["image_label"].nunique()<2: continue
    grand=y.mean(); total=np.mean((y-grand)**2); means=q.groupby("image_label")["image_rate_hz"].agg(["mean","size"])
    between=float(np.sum(means["size"]*(means["mean"]-grand)**2)/len(y))
    row=dict(zip(metric_keys,keys)); row.update(mean_rate_hz=float(grand),total_rate_variance_hz2=float(total),identity_rate_variance_hz2=between,image_rms_hz=float(np.sqrt(between)),image_fve=float(between/total) if total>0 else np.nan,n_trials=int(len(y)),n_images=int(len(means)))
    metric_rows.append(row)
image_metrics=pd.DataFrame(metric_rows)
registered_metrics=image_metrics[image_metrics["manually_registered"]].copy()
cell_metrics=registered_metrics.groupby(["subject_id","global_cell_id","depth_group"],observed=True).agg(depth_um=("depth_um","median"),total_rate_variance_hz2=("total_rate_variance_hz2","median"),image_fve=("image_fve","median"),image_rms_hz=("image_rms_hz","median"),n_sessions=("session_id","nunique")).reset_index()
display(image_metrics.head())

In [ ]:
def plot_longitudinal(ax,df,metric,ylabel):
    labels=[s for s in SESSION_ORDER if s in set(df["session_label"].astype(str))]; xm={s:i for i,s in enumerate(labels)}; xx=np.arange(len(labels))
    tracked=df.groupby(["subject_id","global_cell_id"],group_keys=False,observed=True).filter(lambda z:z["session_id"].nunique()>=2)
    for (_,cell),g in tracked.groupby(["subject_id","global_cell_id"],observed=True):
        g=g.assign(x=g["session_label"].map(xm)).dropna(subset=["x"]).sort_values("x")
        if len(g)>1: ax.plot(g["x"],g[metric],color=DEPTH_COLORS[str(g["depth_group"].iloc[0])],lw=.8,alpha=.12,zorder=1)
    for group,g in df.groupby("depth_group",observed=True):
        q=g.groupby("session_label")[metric].agg(median="median",q25=lambda z:np.nanquantile(z,.25),q75=lambda z:np.nanquantile(z,.75)).reindex(labels)
        valid=q["median"].notna().to_numpy(); xq=xx[valid]
        if not valid.any(): continue
        c=DEPTH_COLORS[str(group)]
        ax.fill_between(xq,q["q25"].to_numpy()[valid],q["q75"].to_numpy()[valid],color=c,alpha=.12,lw=0,zorder=2)
        ax.plot(xq,q["median"].to_numpy()[valid],"-o",color=c,ms=7,mec="black",mew=.8,lw=3,zorder=3,label=str(group))
    if "A2" in xm and "B0" in xm: ax.axvline((xm["A2"]+xm["B0"])/2,color=".65",lw=1,ls=":")
    ax.set(xticks=xx,xticklabels=labels,xlabel="Session",ylabel=ylabel); finish_axis(ax)

def add_depth_legend(ax,loc="best"):
    ax.legend(handles=[Line2D([0],[0],color=DEPTH_COLORS[g],lw=3,marker="o",mec="black",mew=.6,label=g) for g in DEPTH_GROUP_ORDER],title="Depth",frameon=False,fontsize=9,loc=loc)

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(9.2,3.6))
order=[g for g in DEPTH_GROUP_ORDER if g in set(cell_metrics["depth_group"].astype(str))]
sns.violinplot(data=cell_metrics,x="depth_group",y="total_rate_variance_hz2",order=order,palette=DEPTH_COLORS,inner=None,width=.5,linewidth=1.5,ax=axs[0])
for coll in axs[0].collections: coll.set_alpha(.25); coll.set_edgecolor("black")
rng=np.random.default_rng(8)
xpos=np.array([order.index(str(x)) for x in cell_metrics["depth_group"]],float)+rng.uniform(-.12,.12,len(cell_metrics))
axs[0].scatter(xpos,cell_metrics["total_rate_variance_hz2"],s=42,c=[DEPTH_COLORS[str(x)] for x in cell_metrics["depth_group"]],edgecolor="black",linewidth=.6,zorder=3)
axs[0].set(xlabel="Depth bin",ylabel="Median trial-wise firing-rate\nvariance across sessions (Hz²)",title="Aggregate image-response variability"); finish_axis(axs[0])

plot_longitudinal(axs[1],registered_metrics,"total_rate_variance_hz2","Trial-wise firing-rate variance (Hz²)")
axs[1].set_title("Image-response variability across sessions"); add_depth_legend(axs[1])
fig.tight_layout(); save_figure(fig,os.path.join(SAVE_PATH,"image_trial_variance"),formats=[".pdf"],dpi=300); plt.show()

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(9.2,3.6))
plot_longitudinal(axs[0],registered_metrics,"image_fve","Stimulus identity FVE")
plot_longitudinal(axs[1],registered_metrics,"image_rms_hz","RMS image-identity modulation (Hz)")
axs[0].axhline(0,color=".6",lw=1,ls="--"); axs[1].axhline(0,color=".6",lw=1,ls="--")
axs[0].set_title("Variance explained by image identity"); axs[1].set_title("Image-identity response magnitude")
add_depth_legend(axs[1]); fig.tight_layout()
save_figure(fig,os.path.join(SAVE_PATH,"image_identity_fve_rms"),formats=[".pdf"],dpi=300); plt.show()
display(
    image_metrics
    .sort_values(["subject_id","session_order","depth_um","global_cell_id"])
    [["subject_id","session_label","global_cell_id","depth_group","mean_rate_hz","total_rate_variance_hz2","identity_rate_variance_hz2","image_fve","image_rms_hz","n_trials","n_images"]]
)

# 3. Image-sequence dynamics
Slope is fit to mean 0–250 ms firing rate across the first six presented images after a change; sequences are truncated at the first omission.

In [ ]:
MAX_SEQUENCE_PRESENTATIONS = 15
MIN_EPOCHS_PER_POSITION = 0
MIN_SEQUENCE_POSITIONS = 4
REQUIRE_COMPLETE_SEQUENCE_BLOCK = True
PREFERRED_SLOPE_MODE = "max"   # largest signed slope on A0 or B0

In [ ]:
seq_events=[]
for sid,g in events.sort_values(["session_id","event_id"]).groupby("session_id",observed=True):
    q=g.copy(); q["change_epoch"]=q["is_change"].astype(bool).cumsum()
    targets=q.loc[q["is_change"].astype(bool) & q["image_label"].notna(),["change_epoch","image_label"]].drop_duplicates("change_epoch").set_index("change_epoch")["image_label"]
    q["sequence_image"]=q["change_epoch"].map(targets)
    q["presented_target"]=(~q["is_omission"].astype(bool)) & q["image_label"].eq(q["sequence_image"]) & q["sequence_image"].notna()
    q["sequence_position"]=q.groupby("change_epoch",observed=True)["presented_target"].cumsum()
    q["omission_before"]=q.groupby("change_epoch",observed=True)["is_omission"].transform(lambda x:x.astype(bool).shift(fill_value=False).cummax())
    q=q[(q["change_epoch"]>0) & q["retained_image_window"].astype(bool) & q["presented_target"] & ~q["omission_before"] & q["sequence_position"].between(1,MAX_SEQUENCE_PRESENTATIONS)]
    seq_events.append(q[["subject_id","session_id","session_label","session_order","event_id","onset_sec","change_epoch","sequence_image","sequence_position"]])
sequence_events=pd.concat(seq_events,ignore_index=True)

seq_trials=[]
for session in sessions[sessions["session_label"].astype(str).isin(SESSION_ORDER)].itertuples(index=False):
    sid=str(session.session_id); ev=sequence_events[sequence_events["session_id"].astype(str)==sid].copy(); onsets=ev["onset_sec"].to_numpy(float)
    for r in rois[(rois["session_id"].astype(str)==sid) & rois["included"].astype(bool)].itertuples(index=False):
        st=spike_lookup.get((sid,int(r.dmd),int(r.roi)),np.array([],float))
        evr=ev.copy(); evr["sequence_rate_hz"]=(np.searchsorted(st,onsets+IMAGE_WINDOW_S[1],side="left")-np.searchsorted(st,onsets+IMAGE_WINDOW_S[0],side="left"))/(IMAGE_WINDOW_S[1]-IMAGE_WINDOW_S[0])
        evr["dmd"]=int(r.dmd); evr["roi"]=int(r.roi); evr["cell_id"]=str(r.cell_id); evr["global_cell_id"]=str(getattr(r,"global_cell_id","")); evr["manually_registered"]=bool(getattr(r,"manually_registered",False)); evr["depth_um"]=float(r.depth_um); evr["depth_group"]=str(r.depth_group)
        seq_trials.append(evr)
sequence_trial_df=pd.concat(seq_trials,ignore_index=True)

seq_keys=["subject_id","session_id","session_label","session_order","dmd","roi","cell_id","global_cell_id","manually_registered","depth_um","depth_group","sequence_image","sequence_position"]
sequence_position_df=(sequence_trial_df.groupby(seq_keys,observed=True,dropna=False)
    .agg(mean_rate_hz=("sequence_rate_hz","mean"),sem_rate_hz=("sequence_rate_hz",lambda x:np.nanstd(x,ddof=1)/np.sqrt(np.isfinite(x).sum()) if np.isfinite(x).sum()>1 else np.nan),n_epochs=("change_epoch","nunique"))
    .reset_index())
sequence_position_df=sequence_position_df[sequence_position_df["n_epochs"]>=MIN_EPOCHS_PER_POSITION]

slope_keys=["subject_id","session_id","session_label","session_order","dmd","roi","cell_id","global_cell_id","manually_registered","depth_um","depth_group","sequence_image"]
rows=[]
for keys,g in sequence_position_df.groupby(slope_keys,observed=True,dropna=False):
    g=g.sort_values("sequence_position")
    if len(g)<MIN_SEQUENCE_POSITIONS or 1 not in set(g["sequence_position"]): continue
    x=g["sequence_position"].to_numpy(float); y=g["mean_rate_hz"].to_numpy(float); slope,intercept=np.polyfit(x,y,1); pred=intercept+slope*x
    ss_res=np.sum((y-pred)**2); ss_tot=np.sum((y-y.mean())**2)
    row=dict(zip(slope_keys,keys)); row.update(sequence_slope_hz_per_presentation=float(slope),sequence_intercept_hz=float(intercept),sequence_r2=float(1-ss_res/ss_tot) if ss_tot>0 else np.nan,n_positions=int(len(g)),min_epochs_per_position=int(g["n_epochs"].min()),rate_first_hz=float(g.iloc[0]["mean_rate_hz"]),rate_last_hz=float(g.iloc[-1]["mean_rate_hz"]))
    rows.append(row)
sequence_slopes=pd.DataFrame(rows)
print(f"{len(sequence_events):,} sequence presentations · {len(sequence_slopes):,} neuron × image × session slopes")
display(sequence_slopes.head())

## Example ROI

In [ ]:
cand=sequence_slopes[sequence_slopes["session_label"].astype(str).eq("A0")].copy()
if cand.empty: cand=sequence_slopes.copy()
ex=cand.loc[cand["sequence_slope_hz_per_presentation"].idxmax()]
m=(sequence_position_df["session_id"].astype(str).eq(str(ex["session_id"])) & sequence_position_df["dmd"].eq(ex["dmd"]) & sequence_position_df["roi"].eq(ex["roi"]) & sequence_position_df["sequence_image"].eq(ex["sequence_image"]))
mean_ex=sequence_position_df[m].sort_values("sequence_position")
mraw=(sequence_trial_df["session_id"].astype(str).eq(str(ex["session_id"])) & sequence_trial_df["dmd"].eq(ex["dmd"]) & sequence_trial_df["roi"].eq(ex["roi"]) & sequence_trial_df["sequence_image"].eq(ex["sequence_image"]) & sequence_trial_df["sequence_position"].isin(mean_ex["sequence_position"]))
raw_ex=sequence_trial_df[mraw]; c=DEPTH_COLORS[str(ex["depth_group"])]; rng=np.random.default_rng(8)

fig,ax=plt.subplots(figsize=(4.4,3.5))
# ax.scatter(raw_ex["sequence_position"]+rng.uniform(-.10,.10,len(raw_ex)),raw_ex["sequence_rate_hz"],s=14,color=c,alpha=.12,edgecolor="none")
ax.errorbar(mean_ex["sequence_position"],mean_ex["mean_rate_hz"],yerr=mean_ex["sem_rate_hz"],fmt="o-",color=c,lw=3,ms=7,mec="black",mew=.8,capsize=2,zorder=3)
xx=np.array([mean_ex["sequence_position"].min(),mean_ex["sequence_position"].max()]); ax.plot(xx,ex["sequence_intercept_hz"]+ex["sequence_slope_hz_per_presentation"]*xx,color="black",lw=2,ls="--")
ax.set(xlabel="Presentation after image change",ylabel="Image-epoch firing rate (Hz)",title=f'Mouse {ex["subject_id"]} · {ex["session_label"]} · {ex["depth_group"]}\n{ex["sequence_image"]} · slope = {ex["sequence_slope_hz_per_presentation"]:.2f} Hz/presentation')
ax.set_xticks(range(1,MAX_SEQUENCE_PRESENTATIONS+1)); finish_axis(ax); fig.tight_layout()
save_figure(fig,os.path.join(SAVE_PATH,"sequence_example_roi"),formats=[".pdf"],dpi=300); plt.show()

## Sequence slope by session and depth

In [ ]:
fig,ax=plt.subplots(figsize=(5.8,3.8)); xmap={s:i for i,s in enumerate(SESSION_ORDER)}; offsets={g:o for g,o in zip(DEPTH_GROUP_ORDER,[-.18,0,.18])}; rng=np.random.default_rng(8)
for group,g in sequence_slopes.groupby("depth_group",observed=True):
    c=DEPTH_COLORS[str(group)]
    xx=g["session_label"].map(xmap).to_numpy(float)+offsets.get(str(group),0)+rng.uniform(-.055,.055,len(g))
    ax.scatter(xx,g["sequence_slope_hz_per_presentation"],s=19,color=c,alpha=.25,edgecolor="none")
    q=g.groupby("session_label")["sequence_slope_hz_per_presentation"].agg(median="median",q25=lambda z:np.nanquantile(z,.25),q75=lambda z:np.nanquantile(z,.75)).reindex(SESSION_ORDER)
    valid=q["median"].notna().to_numpy(); xm=np.arange(len(SESSION_ORDER))[valid]+offsets.get(str(group),0)
    ax.fill_between(xm,q["q25"].to_numpy()[valid],q["q75"].to_numpy()[valid],color=c,alpha=.12,lw=0)
    ax.plot(xm,q["median"].to_numpy()[valid],"-o",color=c,lw=2.8,ms=6,mec="black",mew=.7,label=str(group))
ax.axhline(0,color=".55",lw=1,ls="--"); ax.axvline(2.5,color=".7",lw=1,ls=":")
ax.set(xticks=np.arange(len(SESSION_ORDER)),xticklabels=SESSION_ORDER,xlabel="Session",ylabel="Sequence slope (Hz / presentation)",title="Image-sequence slope across sessions")
finish_axis(ax); add_depth_legend(ax)
fig.tight_layout(); save_figure(fig,os.path.join(SAVE_PATH,"sequence_slopes_by_depth"),formats=[".pdf"],dpi=300); plt.show()

## Longitudinal change in each neuron's strongest image-sequence slope

In [ ]:
def preferred_sequence_block(df,labels):
    start=labels[0]; q=df[df["session_label"].astype(str).isin(labels) & df["manually_registered"].astype(bool) & df["global_cell_id"].astype(str).ne("")].copy()
    s=q[q["session_label"].astype(str).eq(start)].copy()
    s["rank_value"]=s["sequence_slope_hz_per_presentation"].abs() if PREFERRED_SLOPE_MODE=="max_abs" else s["sequence_slope_hz_per_presentation"]
    pref=s.sort_values("rank_value").drop_duplicates(["subject_id","global_cell_id"],keep="last")[["subject_id","global_cell_id","sequence_image","sequence_slope_hz_per_presentation","depth_group"]].rename(columns={"sequence_image":"preferred_image","sequence_slope_hz_per_presentation":"start_slope","depth_group":"start_depth_group"})
    q=q.merge(pref,on=["subject_id","global_cell_id"],how="inner"); q=q[q["sequence_image"].eq(q["preferred_image"])]
    if REQUIRE_COMPLETE_SEQUENCE_BLOCK:
        complete=q.groupby(["subject_id","global_cell_id"],observed=True)["session_label"].nunique(); keep=complete[complete==len(labels)].index
        q=q.set_index(["subject_id","global_cell_id"]).loc[keep].reset_index() if len(keep) else q.iloc[0:0]
    q["delta_sequence_slope"]=q["sequence_slope_hz_per_presentation"]-q["start_slope"]; q["block"]=f"{labels[0]}→{labels[-1]}"
    return q

preferred_A=preferred_sequence_block(sequence_slopes,["A0","A1","A2"])
preferred_B=preferred_sequence_block(sequence_slopes,["B0","B1","B2"])
preferred_sequence_change=pd.concat([preferred_A,preferred_B],ignore_index=True)
display(preferred_sequence_change[["subject_id","global_cell_id","block","preferred_image","session_label","start_depth_group","sequence_slope_hz_per_presentation","delta_sequence_slope"]].head())

In [ ]:
blocks=[("A0→A2",["A0","A1","A2"]),("B0→B2",["B0","B1","B2"])]
fig,axs=plt.subplots(1,2,figsize=(8.4,3.5),sharey=True)
for ax,(block,labels) in zip(axs,blocks):
    q=preferred_sequence_change[preferred_sequence_change["block"].eq(block)].copy(); xm={s:i for i,s in enumerate(labels)}
    for (_,cell),g in q.groupby(["subject_id","global_cell_id"],observed=True):
        g=g.assign(x=g["session_label"].map(xm)).dropna(subset=["x"]).sort_values("x"); color=DEPTH_COLORS[str(g["start_depth_group"].iloc[0])]
#         ax.plot(g["x"],g["delta_sequence_slope"],"-o",color=color,lw=1,alpha=.22,ms=4)
    for group,g in q.groupby("start_depth_group",observed=True):
        s=g.groupby("session_label")["delta_sequence_slope"].agg(median="median",q25=lambda z:np.nanquantile(z,.25),q75=lambda z:np.nanquantile(z,.75)).reindex(labels)
        valid=s["median"].notna().to_numpy(); xx=np.arange(len(labels))[valid]; color=DEPTH_COLORS[str(group)]
        ax.fill_between(xx,s["q25"].to_numpy()[valid],s["q75"].to_numpy()[valid],color=color,alpha=.12,lw=0)
        ax.plot(xx,s["median"].to_numpy()[valid],"-o",color=color,lw=3,ms=7,mec="black",mew=.8,label=str(group),zorder=3)
    ax.axhline(0,color=".55",lw=1,ls="--"); ax.set(xticks=np.arange(len(labels)),xticklabels=labels,xlabel="Session",title=block); finish_axis(ax)
axs[0].set_ylabel("Δ preferred-image sequence slope\nfrom first day (Hz / presentation)")
add_depth_legend(axs[1])
fig.tight_layout(); save_figure(fig,os.path.join(SAVE_PATH,"preferred_sequence_slope_change"),formats=[".pdf"],dpi=300); plt.show()

# 4. Image-change responses
*Next section: peri-change spike-rate response magnitude, timing, and longitudinal dynamics.*

# 5. Omission responses
*Next section: peri-omission ramping, onset/offset responses, magnitude, timing, and longitudinal dynamics.*